# MSDeepAMR Fine-Tuning in PyTorch via ONNX Conversion

**Goal**: Transfer the pretrained MSDeepAMR 1D-CNN weights (stored as Keras `.h5` files)
into PyTorch and reproduce — and extend — the three fine-tuning strategies originally
implemented in TensorFlow.

## Pipeline overview
```
Keras .h5  ──► tf2onnx ──► .onnx  ──► onnx2torch ──► nn.Module
                                                         │
                               ┌─────────────────────────┤
                               │  Strategy A: Zero-shot  │
                               │  Strategy B: Full FT    │
                               │  Strategy C: Partial FT │
                               └─────────────────────────┘
```

## Why ONNX → onnx2torch instead of raw ONNX Runtime?
ONNX Runtime is inference-only — it has no autograd graph, so gradients cannot
flow through it during training. `onnx2torch` (ENOT-AutoDL) reconstructs the
computation graph as a genuine `torch.nn.Module` with real `nn.Parameter` tensors,
giving us `.parameters()`, `.train()`, `.eval()`, and full backprop.

## Dataset
DRIAMS C2018 — 9,658 MALDI-TOF mass spectra, 6,000 m/z intensity bins per spectrum.
Three pathogen–drug pairs evaluated:
- *Escherichia coli* / Ceftriaxone
- *Klebsiella pneumoniae* / Ceftriaxone
- *Staphylococcus aureus* / Oxacillin

## Cell 1 — Install dependencies
Run once per runtime. All packages are pinned for reproducibility.

In [ ]:
# ── Install all required packages ────────────────────────────────────────────
# tf2onnx  : converts Keras / TF SavedModel → ONNX
# onnx2torch: converts ONNX graph → trainable torch.nn.Module
# onnxruntime: needed by onnx2torch internally for shape inference

%pip install --quiet tf2onnx==1.16.1
%pip install --quiet onnx2torch==1.5.15
%pip install --quiet onnxruntime==1.19.2
%pip install --quiet onnx==1.16.2
%pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu118
%pip install --quiet scikit-learn matplotlib pandas numpy

print("All packages installed.")

## Cell 2 — Imports & version check

In [ ]:
import os
import copy
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import onnx
import tf2onnx
from onnx2torch import convert as onnx2torch_convert

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import sklearn
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, balanced_accuracy_score,
    roc_curve, precision_recall_curve,
    auc as sklearn_auc
)

print(f"PyTorch       : {torch.__version__}")
print(f"ONNX          : {onnx.__version__}")
print(f"tf2onnx       : {tf2onnx.__version__}")
print(f"scikit-learn  : {sklearn.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device  : {DEVICE}")

## Cell 3 — Paths & model registry

In [ ]:
# ── Adjust BASE_DIR to your environment ──────────────────────────────────────
# Google Colab with Google Drive:
#   BASE_DIR = '/content/drive/MyDrive/MSDeepAMR'
# Local machine:
#   BASE_DIR = os.path.expanduser('~/Documents/deepAMR/DL-AMR')

BASE_DIR  = '/content/drive/MyDrive/MSDeepAMR'   # ← change if running locally
C2018_DIR = os.path.join(BASE_DIR, 'data', 'C2018')
ONNX_DIR  = os.path.join(BASE_DIR, 'onnx_models')  # ONNX exports stored here
PT_DIR    = os.path.join(BASE_DIR, 'pt_models')     # fine-tuned .pt files saved here

os.makedirs(ONNX_DIR, exist_ok=True)
os.makedirs(PT_DIR,   exist_ok=True)

# ── Pretrained Keras model files (284 MB each) ───────────────────────────────
H5_MODELS = {
    'e_coli_ceftriaxone'     : os.path.join(BASE_DIR, 'e_coli_ceftriaxone.h5'),
    'k_pneumoniae_ceftriaxone': os.path.join(BASE_DIR, 'k_pneumoniae_ceftriaxone.h5'),
    's_aureus_oxacillin'     : os.path.join(BASE_DIR, 's_aureus_oxacillin.h5'),
}

MODEL_META = {
    'e_coli_ceftriaxone'     : {'species': 'Escherichia coli',     'drug': 'Ceftriaxone'},
    'k_pneumoniae_ceftriaxone': {'species': 'Klebsiella pneumoniae','drug': 'Ceftriaxone'},
    's_aureus_oxacillin'     : {'species': 'Staphylococcus aureus', 'drug': 'Oxacillin'},
}

print("Paths configured:")
for name, path in H5_MODELS.items():
    status = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f"  [{status}]  {name}")

## Cell 4 — Fine-tuning hyper-parameters

All hyper-parameters are intentionally kept identical to the TensorFlow notebook
so results are directly comparable.

In [ ]:
# ── Learning rates (matching the TF notebook) ─────────────────────────────────
LR_FULL_FINETUNE    = 1e-5   # all layers unfrozen
LR_PARTIAL_UNFREEZE = 5e-6   # conv blocks 3 & 4 + dense head unfrozen

# ── Training schedule ────────────────────────────────────────────────────────
EPOCHS_FINETUNE = 50         # upper bound; EarlyStopping cuts this short
BATCH_SIZE      = 10
PATIENCE        = 5          # EarlyStopping patience (epochs)
LR_PATIENCE     = 3          # ReduceLROnPlateau patience
LR_FACTOR       = 0.1        # LR reduction factor
MIN_LR          = 1e-7

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Result accumulator ───────────────────────────────────────────────────────
ALL_RESULTS = []

print("Hyper-parameters set.")

---
# Part 1 — ONNX Conversion

Each `.h5` Keras model is exported to ONNX format using `tf2onnx`.  
We use **opset 13** — the broadest opset supported by both `tf2onnx` and `onnx2torch`
for 1D-CNN operations (Conv, BatchNorm, Relu, MaxPool, Gemm).

## Cell 5 — Mount Google Drive (Colab only)
Skip this cell if running locally.

In [ ]:
# Uncomment the two lines below if running in Google Colab
# from google.colab import drive
# drive.mount('/content/drive')

## Cell 6 — Convert .h5 → .onnx

TensorFlow is imported **only in this section** to load the `.h5` weights and run the
export. After conversion is done, TF is no longer needed — the rest of the notebook
is pure PyTorch.

In [ ]:
import tensorflow as tf
import tf2onnx
import tf2onnx.convert

OPSET = 13   # ONNX opset version — 13 gives best coverage for Conv1D + BatchNorm

ONNX_PATHS = {}  # filled in the loop below

for model_name, h5_path in H5_MODELS.items():
    onnx_path = os.path.join(ONNX_DIR, f"{model_name}.onnx")
    ONNX_PATHS[model_name] = onnx_path

    if os.path.exists(onnx_path):
        print(f"[SKIP] {model_name}.onnx already exists — skipping conversion.")
        continue

    print(f"\n[Converting] {model_name} ...")

    # Load Keras model
    keras_model = tf.keras.models.load_model(h5_path, compile=False)

    # Derive the input signature: shape (None, 6000, 1) — channels last for Keras Conv1D
    input_signature = [
        tf.TensorSpec(
            shape=(None, 6000, 1),
            dtype=tf.float32,
            name='input'
        )
    ]

    # Convert to ONNX
    model_proto, _ = tf2onnx.convert.from_keras(
        keras_model,
        input_signature=input_signature,
        opset=OPSET,
        output_path=onnx_path
    )

    print(f"  Saved → {onnx_path}")
    print(f"  ONNX opset : {model_proto.opset_import[0].version}")
    print(f"  Graph nodes: {len(model_proto.graph.node)}")

    # Free Keras model from memory before next iteration
    del keras_model
    tf.keras.backend.clear_session()

print("\nAll conversions complete.")

## Cell 7 — Validate ONNX graphs
Run a quick structural check on each ONNX file before loading into PyTorch.

In [ ]:
import onnx

for model_name, onnx_path in ONNX_PATHS.items():
    model_proto = onnx.load(onnx_path)
    try:
        onnx.checker.check_model(model_proto)
        print(f"[OK]  {model_name} — graph is valid, {len(model_proto.graph.node)} nodes")
    except onnx.checker.ValidationError as e:
        print(f"[FAIL] {model_name} — {e}")

## Cell 8 — Load ONNX → PyTorch nn.Module

`onnx2torch.convert()` walks the ONNX computation graph and reconstructs each
operation as a native `torch.nn` layer with `nn.Parameter` weights.  
The returned object is a standard `nn.Module` — you can call `.parameters()`,
`.state_dict()`, `.train()`, `.eval()`, and pass it to any optimizer.

In [ ]:
from onnx2torch import convert as onnx2torch_convert

PT_MODELS = {}  # model_name → nn.Module on DEVICE

for model_name, onnx_path in ONNX_PATHS.items():
    print(f"Loading {model_name} ...")
    onnx_proto   = onnx.load(onnx_path)
    torch_model  = onnx2torch_convert(onnx_proto)
    torch_model  = torch_model.to(DEVICE)
    torch_model.eval()  # start in eval mode (inference default)

    PT_MODELS[model_name] = torch_model

    # Count parameters
    total  = sum(p.numel() for p in torch_model.parameters())
    print(f"  Parameters : {total:,}")

print("\nAll PyTorch models loaded.")

## Cell 9 — Inspect layer names

We print the named modules so we can target the right layers for freezing
in the partial-unfreeze strategy. Run this for one model — the architecture
is identical across all three.

In [ ]:
sample_model = PT_MODELS['e_coli_ceftriaxone']

print("Named modules in the converted PyTorch model:")
print("-" * 60)
for idx, (name, module) in enumerate(sample_model.named_modules()):
    if name:  # skip the root module itself
        param_count = sum(p.numel() for p in module.parameters(recurse=False))
        print(f"  [{idx:>3}]  {name:<50}  {type(module).__name__:<25}  params={param_count:>10,}")

print("-" * 60)
print("\nNOTE: Use the layer names above to set freeze boundaries in the partial-unfreeze strategy.")

---
# Part 2 — Data Pipeline in PyTorch

We replicate the same `build_splits()` logic from the TF notebook, but wrap
the arrays in a PyTorch `Dataset` and `DataLoader`.

**Key difference from TF**: Keras `Conv1D` expects `(N, Length, Channels)` 
(channels-last). PyTorch `Conv1d` expects `(N, Channels, Length)` (channels-first).  
The ONNX graph was exported from Keras, so `onnx2torch` already handles the
transpose internally — we feed `(N, 6000, 1)` tensors just as we did in TF.

## Cell 10 — Load raw C2018 arrays

In [ ]:
# ── Raw spectra ───────────────────────────────────────────────────────────────
raw_spectra = np.load(os.path.join(C2018_DIR, 'rawSpectra_data.npy'), allow_pickle=True)
labels_df   = pd.read_csv(os.path.join(C2018_DIR, 'combined_long_table.csv'))
splits_df   = pd.read_csv(os.path.join(C2018_DIR, 'data_splits.csv'))

print(f"rawSpectra shape : {raw_spectra.shape}   dtype={raw_spectra.dtype}")
print(f"labels_df shape  : {labels_df.shape}")
print(f"splits_df shape  : {splits_df.shape}")
print(f"Spectra range    : [{raw_spectra.min():.4f}, {raw_spectra.max():.4f}]")

# ── sample_id → row-index mapping ────────────────────────────────────────────
# The npy file rows align with splits_df sorted by sample_id
ordered_ids = sorted(splits_df['sample_id'].unique())
id_to_row   = {sid: i for i, sid in enumerate(ordered_ids)}

assert len(ordered_ids) == raw_spectra.shape[0], \
    "Mismatch between unique sample IDs and rawSpectra rows!"
print(f"\n✓ Alignment check passed — {len(ordered_ids):,} samples")

## Cell 11 — SpectraDataset class

In [ ]:
class SpectraDataset(Dataset):
    """
    PyTorch Dataset wrapping MALDI-TOF mass spectra and binary AMR labels.

    Each item is:
      x : FloatTensor  shape (6000, 1)  — one spectrum, channels-last
      y : FloatTensor  scalar           — 0 (Susceptible) or 1 (Resistant)

    Channels-last is preserved to match the ONNX graph's expected input layout.
    """

    def __init__(self, spectra: np.ndarray, labels: np.ndarray):
        """
        Parameters
        ----------
        spectra : np.ndarray  shape (N, 6000)
        labels  : np.ndarray  shape (N,)  — float32 binary
        """
        # Reshape: (N, 6000) → (N, 6000, 1)  [channels last, matching Keras export]
        self.X = torch.from_numpy(
            spectra.astype(np.float32).reshape(-1, 6000, 1)
        )
        self.y = torch.from_numpy(labels.astype(np.float32))

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


print("SpectraDataset defined.")

## Cell 12 — build_splits() helper

In [ ]:
def build_splits(
    species: str,
    drug: str,
    batch_size: int = BATCH_SIZE,
    verbose: bool = True
):
    """
    Build train / val / test DataLoaders for a given species–drug pair.

    The training DataLoader uses WeightedRandomSampler to up-sample the
    minority (Resistant) class, mitigating the class-imbalance problem.

    Returns
    -------
    train_loader, val_loader, test_loader : DataLoader
    X_test_np, y_test_np : raw numpy arrays for sklearn metric computation
    X_train_np, y_train_np : raw numpy arrays (for baseline inference)
    """
    # ── Filter labels to this species & drug ──────────────────────────────────
    mask = (labels_df['species'] == species) & (labels_df['drug'] == drug)
    sub  = labels_df[mask][['sample_id', 'response']].dropna()
    sub  = sub.merge(splits_df, on='sample_id', how='inner')
    sub  = sub[sub['sample_id'].isin(id_to_row)].copy()
    sub['row_idx'] = sub['sample_id'].map(id_to_row)

    def get_arrays(split_name):
        rows = sub[sub['Set'] == split_name]
        X    = raw_spectra[rows['row_idx'].values].astype(np.float32)
        y    = rows['response'].values.astype(np.float32)
        return X, y

    X_train, y_train = get_arrays('train')
    X_val,   y_val   = get_arrays('validation')
    X_test,  y_test  = get_arrays('test')

    # ── Build Datasets ────────────────────────────────────────────────────────
    train_ds = SpectraDataset(X_train, y_train)
    val_ds   = SpectraDataset(X_val,   y_val)
    test_ds  = SpectraDataset(X_test,  y_test)

    # ── WeightedRandomSampler for class-imbalance (training only) ─────────────
    # Computes per-sample weights so each class is seen equally per epoch.
    class_counts  = np.bincount(y_train.astype(int))       # [n_susceptible, n_resistant]
    class_weights = 1.0 / (class_counts + 1e-6)            # inverse frequency
    sample_weights = class_weights[y_train.astype(int)]    # one weight per sample
    sampler = WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(train_ds),
        replacement = True
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler,
                              num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=0, pin_memory=True)

    if verbose:
        print(f"  Species : {species}")
        print(f"  Drug    : {drug}")
        n_res_train = int(y_train.sum())
        n_res_val   = int(y_val.sum())
        n_res_test  = int(y_test.sum())
        print(f"  Train   : {len(y_train):>5} samples | "
              f"Resistant={n_res_train} ({100*y_train.mean():.1f}%)")
        print(f"  Val     : {len(y_val):>5} samples | "
              f"Resistant={n_res_val} ({100*y_val.mean():.1f}%)")
        print(f"  Test    : {len(y_test):>5} samples | "
              f"Resistant={n_res_test} ({100*y_test.mean():.1f}%)")

    return (train_loader, val_loader, test_loader,
            X_test, y_test, X_train, y_train)


print("build_splits() defined.")

---
# Part 3 — Fine-Tuning Strategy Definitions

Three strategies, each returning a configured `nn.Module` ready for training:

| Strategy | Frozen layers | Learning rate |
|---|---|---|
| A — Zero-shot baseline | all (inference only) | — |
| B — Full fine-tune | none | 1e-5 |
| C — Partial unfreeze | Conv blocks 1 & 2 | 5e-6 |

## Cell 13 — Strategy configuration functions

In [ ]:
def get_model_copy(model_name: str) -> nn.Module:
    """
    Returns a deep copy of the converted PyTorch model so each experiment
    starts from identical pretrained weights.
    """
    return copy.deepcopy(PT_MODELS[model_name]).to(DEVICE)


def strategy_full_finetune(model_name: str):
    """
    Strategy B — Full fine-tune.
    Every parameter is trainable. Low LR to avoid catastrophic forgetting.
    """
    model = get_model_copy(model_name)
    for param in model.parameters():
        param.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=LR_FULL_FINETUNE)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Full fine-tune  — trainable params: {trainable:,} / {total:,}")
    return model, optimizer


def strategy_partial_unfreeze(model_name: str):
    """
    Strategy C — Partial unfreeze.

    Freeze the first two convolutional blocks (low-level peak detectors shared
    across all spectra) and allow the deeper blocks + classification head to adapt.

    Freezing logic:
    - Iterate over named modules in order.
    - Count how many Conv / BatchNorm / Pool groups have been seen.
    - Freeze everything in the first two Conv–BN–Pool groups.
    - Leave all subsequent modules trainable.

    Because onnx2torch names layers sequentially (e.g. /Conv_0, /Conv_1, ...)
    we use index-based freezing as a robust fallback that works regardless of
    the exact names assigned by the converter.
    """
    model = get_model_copy(model_name)

    # Collect all top-level named children (direct submodules)
    all_children = list(model.named_children())

    # The architecture has 4 conv blocks × 4 layers (Conv+BN+ReLU+Pool) = 16 children,
    # then Flatten, Dropout, Dense×4 = remaining 6 children.
    # Freeze the first 8 children (= 2 conv blocks).
    FREEZE_UNTIL_INDEX = 8

    for idx, (name, child) in enumerate(all_children):
        trainable = idx >= FREEZE_UNTIL_INDEX
        for param in child.parameters():
            param.requires_grad = trainable

    # Also handle any params directly on the root module
    # (onnx2torch sometimes attaches parameters there)
    for name, param in model.named_parameters():
        # If we haven't set requires_grad yet via children, default to trainable
        # (this is a no-op if all params are covered by named_children)
        pass

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_PARTIAL_UNFREEZE
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    print(f"  Partial unfreeze — trainable: {trainable:,}  frozen: {frozen:,}")
    return model, optimizer


print("Strategy functions defined.")

---
# Part 4 — Training Loop

A single `train_model()` function handles all fine-tuning experiments.
It implements:
- Standard forward → loss → backward → step loop
- **EarlyStopping** on validation loss (patience=5, restore best weights)
- **ReduceLROnPlateau** scheduler (factor=0.1, patience=3)

## Cell 14 — EarlyStopping callback

In [ ]:
class EarlyStopping:
    """
    Monitors a validation metric and stops training when no improvement
    is seen for `patience` consecutive epochs. Restores the best weights
    automatically (equivalent to Keras restore_best_weights=True).
    """

    def __init__(self, patience: int = 5, min_delta: float = 1e-6, mode: str = 'min'):
        self.patience   = patience
        self.min_delta  = min_delta
        self.mode       = mode
        self._best_val  = float('inf') if mode == 'min' else -float('inf')
        self._counter   = 0
        self._best_state = None
        self.stopped_epoch = 0

    def _is_improvement(self, current):
        if self.mode == 'min':
            return current < self._best_val - self.min_delta
        return current > self._best_val + self.min_delta

    def step(self, val_metric, model: nn.Module, epoch: int) -> bool:
        """
        Returns True if training should stop.
        """
        if self._is_improvement(val_metric):
            self._best_val   = val_metric
            self._counter    = 0
            # Deep-copy the state dict so we can restore it later
            self._best_state = copy.deepcopy(model.state_dict())
        else:
            self._counter += 1
            if self._counter >= self.patience:
                self.stopped_epoch = epoch
                return True  # stop
        return False

    def restore_best_weights(self, model: nn.Module):
        if self._best_state is not None:
            model.load_state_dict(self._best_state)


print("EarlyStopping defined.")

## Cell 15 — Core training loop

In [ ]:
def train_model(
    model: nn.Module,
    optimizer: optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = EPOCHS_FINETUNE,
    label: str = ''
) -> dict:
    """
    Run the fine-tuning loop with EarlyStopping and ReduceLROnPlateau.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
              (lists, one value per completed epoch)
    """
    criterion    = nn.BCELoss()      # binary cross-entropy (model outputs sigmoid)
    scheduler    = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=LR_FACTOR,
        patience=LR_PATIENCE, min_lr=MIN_LR
    )
    early_stop   = EarlyStopping(patience=PATIENCE, mode='min')

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, epochs + 1):
        # ── Training phase ────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total   = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)

            optimizer.zero_grad()
            logits = model(X_batch)          # raw output (sigmoid already applied)
            # onnx2torch preserves the final sigmoid, so output ∈ (0,1)
            preds  = logits.squeeze(-1)      # (batch,)
            loss   = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item() * len(y_batch)
            predicted      = (preds.detach() > 0.5).float()
            train_correct += (predicted == y_batch).sum().item()
            train_total   += len(y_batch)

        train_loss /= train_total
        train_acc   = train_correct / train_total

        # ── Validation phase ──────────────────────────────────────────────────
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total   = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(DEVICE)
                y_batch = y_batch.to(DEVICE)
                preds   = model(X_batch).squeeze(-1)
                loss    = criterion(preds, y_batch)
                val_loss    += loss.item() * len(y_batch)
                predicted    = (preds > 0.5).float()
                val_correct += (predicted == y_batch).sum().item()
                val_total   += len(y_batch)

        val_loss /= val_total
        val_acc   = val_correct / val_total

        # ── Logging ───────────────────────────────────────────────────────────
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        # Step scheduler
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"  Epoch {epoch:>3}/{epochs}  "
              f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_acc={val_acc:.4f}  lr={current_lr:.2e}")

        # EarlyStopping check
        if early_stop.step(val_loss, model, epoch):
            print(f"  Early stopping at epoch {epoch} "
                  f"(best val_loss={early_stop._best_val:.4f})")
            break

    early_stop.restore_best_weights(model)
    print(f"  Best weights restored.")
    return history


print("train_model() defined.")

---
# Part 5 — Evaluation Suite

Mirrors the `evaluate_model()` function from the TF notebook exactly — same
metrics, same plots — so results are directly comparable.

## Cell 16 — evaluate_model() and plot_history()

In [ ]:
def get_predictions(model: nn.Module, X_np: np.ndarray) -> np.ndarray:
    """
    Run inference on a raw numpy array.
    Returns sigmoid probabilities as a (N,) numpy array.
    """
    model.eval()
    dataset    = SpectraDataset(X_np, np.zeros(len(X_np)))  # dummy labels
    loader     = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
    all_probs  = []

    with torch.no_grad():
        for X_batch, _ in loader:
            X_batch = X_batch.to(DEVICE)
            probs   = model(X_batch).squeeze(-1).cpu().numpy()
            all_probs.append(probs)

    return np.concatenate(all_probs)


def evaluate_model(
    model: nn.Module,
    X_test: np.ndarray,
    y_test: np.ndarray,
    X_train: np.ndarray,
    y_train: np.ndarray,
    label: str = '',
    plot: bool = True
) -> dict:
    """
    Full evaluation suite — identical structure to the TF notebook.

    Computes:
    - Confusion matrix
    - Balanced accuracy
    - Classification report
    - AUROC and AUPRC (sklearn, from raw probabilities)
    - ROC curve, Precision-Recall curve, Confusion matrix plots
    """
    test_probs = get_predictions(model, X_test)
    y_pred     = (test_probs > 0.5).astype(int)

    # ── Confusion matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_test, y_pred)
    print("\nConfusion matrix:")
    print(cm)

    # ── Balanced accuracy ─────────────────────────────────────────────────────
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    print(f"Balanced accuracy : {bal_acc:.4f}")

    # ── Classification report ─────────────────────────────────────────────────
    print("\nClassification report:")
    report = classification_report(
        y_test, y_pred,
        target_names=['Susceptible (0)', 'Resistant (1)'],
        zero_division=0,
        output_dict=True
    )
    print(classification_report(
        y_test, y_pred,
        target_names=['Susceptible (0)', 'Resistant (1)'],
        zero_division=0
    ))

    # ── AUROC & AUPRC ─────────────────────────────────────────────────────────
    fpr,    tpr,    _  = roc_curve(y_test, test_probs)
    prec_c, rec_c,  _  = precision_recall_curve(y_test, test_probs)
    roc_auc            = sklearn_auc(fpr, tpr)
    prc_auc            = sklearn_auc(rec_c, prec_c)
    random_baseline    = float(y_test.mean())

    print(f"AUROC  (sklearn)  : {roc_auc:.4f}")
    print(f"AUPRC  (sklearn)  : {prc_auc:.4f}  (random baseline = {random_baseline:.3f})")

    # ── Plots ─────────────────────────────────────────────────────────────────
    if plot:
        colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        fig, axes = plt.subplots(1, 3, figsize=(18, 5))

        # Confusion matrix
        ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=['Susceptible', 'Resistant']
        ).plot(ax=axes[0], colorbar=False, cmap='Blues')
        axes[0].set_title(f'{label}\nConfusion Matrix')

        # ROC curve
        axes[1].plot(fpr, tpr, color=colors[0], linewidth=2,
                     label=f'AUROC = {roc_auc:.3f}')
        axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
        axes[1].set_xlabel('False positive rate')
        axes[1].set_ylabel('True positive rate')
        axes[1].set_title(f'{label}\nROC Curve')
        axes[1].legend(loc='lower right')
        axes[1].grid(True, alpha=0.3)
        axes[1].set_aspect('equal')

        # Precision-Recall curve
        axes[2].plot(rec_c, prec_c, color=colors[0], linewidth=2,
                     label=f'AUPRC = {prc_auc:.3f}')
        axes[2].axhline(y=random_baseline, color='k', linestyle='--', linewidth=1,
                        label=f'No-skill ({random_baseline:.2f})')
        axes[2].set_xlabel('Recall')
        axes[2].set_ylabel('Precision')
        axes[2].set_title(f'{label}\nPrecision-Recall Curve')
        axes[2].legend(loc='upper right')
        axes[2].grid(True, alpha=0.3)
        axes[2].set_aspect('equal')

        plt.tight_layout()
        safe_label = label.replace(' ', '_').replace('/', '-')
        plt.savefig(f'{safe_label}_eval.png', bbox_inches='tight')
        plt.show()

    precision_res = report.get('Resistant (1)', {}).get('precision', 0.0)
    recall_res    = report.get('Resistant (1)', {}).get('recall', 0.0)
    accuracy      = report.get('accuracy', 0.0)

    return {
        'label'        : label,
        'balanced_acc' : round(bal_acc,       4),
        'accuracy'     : round(accuracy,      4),
        'precision'    : round(precision_res, 4),
        'recall'       : round(recall_res,    4),
        'auroc_sk'     : round(roc_auc,       4),
        'auprc_sk'     : round(prc_auc,       4),
        'tn'           : int(cm[0, 0]),
        'fp'           : int(cm[0, 1]),
        'fn'           : int(cm[1, 0]),
        'tp'           : int(cm[1, 1]),
    }


def plot_history(history: dict, label: str = ''):
    """Plot train/val loss and accuracy curves from a history dict."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history['train_loss'], label='train loss')
    axes[0].plot(epochs, history['val_loss'],   label='val loss')
    axes[0].set_title(f'{label} — Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Binary cross-entropy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], label='train acc')
    axes[1].plot(epochs, history['val_acc'],   label='val acc')
    axes[1].set_title(f'{label} — Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle(label, fontsize=13)
    plt.tight_layout()
    safe = label.replace(' ', '_').replace('/', '-')
    plt.savefig(f'{safe}_curves.png', bbox_inches='tight')
    plt.show()


print("evaluate_model() and plot_history() defined.")

---
# Part 6 — Experiments

Each organism block runs three experiments:
1. **Experiment A** — Zero-shot baseline (no training)
2. **Experiment B** — Full fine-tune (all layers)
3. **Experiment C** — Partial unfreeze (conv blocks 3 & 4 + dense head)

Results are accumulated in `ALL_RESULTS` and compared in Part 7.

---
## Organism 1 — *Escherichia coli* / Ceftriaxone

In [ ]:
print("=" * 65)
print("ORGANISM : Escherichia coli")
print("DRUG     : Ceftriaxone")
print("=" * 65)

EC_META = MODEL_META['e_coli_ceftriaxone']
(
    ec_train_loader, ec_val_loader, ec_test_loader,
    ec_X_test,  ec_y_test,
    ec_X_train, ec_y_train
) = build_splits(EC_META['species'], EC_META['drug'])

### Experiment 1-A — Zero-shot baseline (*E. coli*)

In [ ]:
print("--- Experiment 1-A : Zero-shot baseline ---")
ec_baseline = get_model_copy('e_coli_ceftriaxone')
ec_baseline.eval()

res = evaluate_model(
    ec_baseline, ec_X_test, ec_y_test,
    ec_X_train,  ec_y_train,
    label='E.coli/Ceftriaxone — Baseline'
)
res['model'] = 'e_coli_ceftriaxone'
res['experiment'] = 'Baseline'
ALL_RESULTS.append(res)

### Experiment 1-B — Full fine-tune (*E. coli*)

In [ ]:
print("--- Experiment 1-B : Full fine-tune ---")
ec_ft_full, ec_opt_full = strategy_full_finetune('e_coli_ceftriaxone')

hist_ec_full = train_model(
    ec_ft_full, ec_opt_full,
    ec_train_loader, ec_val_loader,
    label='E.coli/Ceftriaxone — Full fine-tune'
)
plot_history(hist_ec_full, label='E.coli/Ceftriaxone — Full fine-tune')

res = evaluate_model(
    ec_ft_full, ec_X_test, ec_y_test,
    ec_X_train, ec_y_train,
    label='E.coli/Ceftriaxone — Full fine-tune'
)
res['model'] = 'e_coli_ceftriaxone'
res['experiment'] = 'Full fine-tune'
ALL_RESULTS.append(res)

# Save fine-tuned weights
torch.save(ec_ft_full.state_dict(),
           os.path.join(PT_DIR, 'e_coli_ceftriaxone_full_ft.pt'))
print("Saved: e_coli_ceftriaxone_full_ft.pt")

### Experiment 1-C — Partial unfreeze (*E. coli*)

In [ ]:
print("--- Experiment 1-C : Partial unfreeze ---")
ec_ft_partial, ec_opt_partial = strategy_partial_unfreeze('e_coli_ceftriaxone')

hist_ec_partial = train_model(
    ec_ft_partial, ec_opt_partial,
    ec_train_loader, ec_val_loader,
    label='E.coli/Ceftriaxone — Partial unfreeze'
)
plot_history(hist_ec_partial, label='E.coli/Ceftriaxone — Partial unfreeze')

res = evaluate_model(
    ec_ft_partial, ec_X_test, ec_y_test,
    ec_X_train,    ec_y_train,
    label='E.coli/Ceftriaxone — Partial unfreeze'
)
res['model'] = 'e_coli_ceftriaxone'
res['experiment'] = 'Partial unfreeze'
ALL_RESULTS.append(res)

torch.save(ec_ft_partial.state_dict(),
           os.path.join(PT_DIR, 'e_coli_ceftriaxone_partial_ft.pt'))
print("Saved: e_coli_ceftriaxone_partial_ft.pt")

---
## Organism 2 — *Klebsiella pneumoniae* / Ceftriaxone

In [ ]:
print("=" * 65)
print("ORGANISM : Klebsiella pneumoniae")
print("DRUG     : Ceftriaxone")
print("=" * 65)

KP_META = MODEL_META['k_pneumoniae_ceftriaxone']
(
    kp_train_loader, kp_val_loader, kp_test_loader,
    kp_X_test,  kp_y_test,
    kp_X_train, kp_y_train
) = build_splits(KP_META['species'], KP_META['drug'])

### Experiment 2-A — Zero-shot baseline (*K. pneumoniae*)

In [ ]:
print("--- Experiment 2-A : Zero-shot baseline ---")
kp_baseline = get_model_copy('k_pneumoniae_ceftriaxone')
kp_baseline.eval()

res = evaluate_model(
    kp_baseline, kp_X_test, kp_y_test,
    kp_X_train,  kp_y_train,
    label='K.pneumoniae/Ceftriaxone — Baseline'
)
res['model'] = 'k_pneumoniae_ceftriaxone'
res['experiment'] = 'Baseline'
ALL_RESULTS.append(res)

### Experiment 2-B — Full fine-tune (*K. pneumoniae*)

In [ ]:
print("--- Experiment 2-B : Full fine-tune ---")
kp_ft_full, kp_opt_full = strategy_full_finetune('k_pneumoniae_ceftriaxone')

hist_kp_full = train_model(
    kp_ft_full, kp_opt_full,
    kp_train_loader, kp_val_loader,
    label='K.pneumoniae/Ceftriaxone — Full fine-tune'
)
plot_history(hist_kp_full, label='K.pneumoniae/Ceftriaxone — Full fine-tune')

res = evaluate_model(
    kp_ft_full, kp_X_test, kp_y_test,
    kp_X_train, kp_y_train,
    label='K.pneumoniae/Ceftriaxone — Full fine-tune'
)
res['model'] = 'k_pneumoniae_ceftriaxone'
res['experiment'] = 'Full fine-tune'
ALL_RESULTS.append(res)

torch.save(kp_ft_full.state_dict(),
           os.path.join(PT_DIR, 'k_pneumoniae_ceftriaxone_full_ft.pt'))
print("Saved: k_pneumoniae_ceftriaxone_full_ft.pt")

### Experiment 2-C — Partial unfreeze (*K. pneumoniae*)

In [ ]:
print("--- Experiment 2-C : Partial unfreeze ---")
kp_ft_partial, kp_opt_partial = strategy_partial_unfreeze('k_pneumoniae_ceftriaxone')

hist_kp_partial = train_model(
    kp_ft_partial, kp_opt_partial,
    kp_train_loader, kp_val_loader,
    label='K.pneumoniae/Ceftriaxone — Partial unfreeze'
)
plot_history(hist_kp_partial, label='K.pneumoniae/Ceftriaxone — Partial unfreeze')

res = evaluate_model(
    kp_ft_partial, kp_X_test, kp_y_test,
    kp_X_train,    kp_y_train,
    label='K.pneumoniae/Ceftriaxone — Partial unfreeze'
)
res['model'] = 'k_pneumoniae_ceftriaxone'
res['experiment'] = 'Partial unfreeze'
ALL_RESULTS.append(res)

torch.save(kp_ft_partial.state_dict(),
           os.path.join(PT_DIR, 'k_pneumoniae_ceftriaxone_partial_ft.pt'))
print("Saved: k_pneumoniae_ceftriaxone_partial_ft.pt")

---
## Organism 3 — *Staphylococcus aureus* / Oxacillin

In [ ]:
print("=" * 65)
print("ORGANISM : Staphylococcus aureus")
print("DRUG     : Oxacillin")
print("=" * 65)

SA_META = MODEL_META['s_aureus_oxacillin']
(
    sa_train_loader, sa_val_loader, sa_test_loader,
    sa_X_test,  sa_y_test,
    sa_X_train, sa_y_train
) = build_splits(SA_META['species'], SA_META['drug'])

### Experiment 3-A — Zero-shot baseline (*S. aureus*)

In [ ]:
print("--- Experiment 3-A : Zero-shot baseline ---")
sa_baseline = get_model_copy('s_aureus_oxacillin')
sa_baseline.eval()

res = evaluate_model(
    sa_baseline, sa_X_test, sa_y_test,
    sa_X_train,  sa_y_train,
    label='S.aureus/Oxacillin — Baseline'
)
res['model'] = 's_aureus_oxacillin'
res['experiment'] = 'Baseline'
ALL_RESULTS.append(res)

### Experiment 3-B — Full fine-tune (*S. aureus*)

In [ ]:
print("--- Experiment 3-B : Full fine-tune ---")
sa_ft_full, sa_opt_full = strategy_full_finetune('s_aureus_oxacillin')

hist_sa_full = train_model(
    sa_ft_full, sa_opt_full,
    sa_train_loader, sa_val_loader,
    label='S.aureus/Oxacillin — Full fine-tune'
)
plot_history(hist_sa_full, label='S.aureus/Oxacillin — Full fine-tune')

res = evaluate_model(
    sa_ft_full, sa_X_test, sa_y_test,
    sa_X_train, sa_y_train,
    label='S.aureus/Oxacillin — Full fine-tune'
)
res['model'] = 's_aureus_oxacillin'
res['experiment'] = 'Full fine-tune'
ALL_RESULTS.append(res)

torch.save(sa_ft_full.state_dict(),
           os.path.join(PT_DIR, 's_aureus_oxacillin_full_ft.pt'))
print("Saved: s_aureus_oxacillin_full_ft.pt")

### Experiment 3-C — Partial unfreeze (*S. aureus*)

In [ ]:
print("--- Experiment 3-C : Partial unfreeze ---")
sa_ft_partial, sa_opt_partial = strategy_partial_unfreeze('s_aureus_oxacillin')

hist_sa_partial = train_model(
    sa_ft_partial, sa_opt_partial,
    sa_train_loader, sa_val_loader,
    label='S.aureus/Oxacillin — Partial unfreeze'
)
plot_history(hist_sa_partial, label='S.aureus/Oxacillin — Partial unfreeze')

res = evaluate_model(
    sa_ft_partial, sa_X_test, sa_y_test,
    sa_X_train,    sa_y_train,
    label='S.aureus/Oxacillin — Partial unfreeze'
)
res['model'] = 's_aureus_oxacillin'
res['experiment'] = 'Partial unfreeze'
ALL_RESULTS.append(res)

torch.save(sa_ft_partial.state_dict(),
           os.path.join(PT_DIR, 's_aureus_oxacillin_partial_ft.pt'))
print("Saved: s_aureus_oxacillin_partial_ft.pt")

---
# Part 7 — Summary & Comparison

## Cell 17 — Results table

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)

col_order = [
    'model', 'experiment',
    'balanced_acc', 'accuracy',
    'precision', 'recall',
    'auroc_sk', 'auprc_sk',
    'tp', 'fp', 'tn', 'fn'
]
results_df = results_df[[c for c in col_order if c in results_df.columns]]

print("Full results table (PyTorch pipeline):")
display(results_df.set_index(['model', 'experiment']))

results_df.to_csv('finetune_pytorch_results.csv', index=False)
print("\nSaved → finetune_pytorch_results.csv")

## Cell 18 — Visual comparison bar charts

In [ ]:
import matplotlib.ticker as mticker

results_df = results_df.drop_duplicates(subset=['experiment', 'model'], keep='last')

pivot_auc    = results_df.pivot(index='experiment', columns='model', values='auroc_sk')
pivot_bal    = results_df.pivot(index='experiment', columns='model', values='balanced_acc')
pivot_recall = results_df.pivot(index='experiment', columns='model', values='recall')

exp_order = ['Baseline', 'Full fine-tune', 'Partial unfreeze']
pivot_auc    = pivot_auc.reindex(exp_order)
pivot_bal    = pivot_bal.reindex(exp_order)
pivot_recall = pivot_recall.reindex(exp_order)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, pivot, title, ylabel in zip(
    axes,
    [pivot_auc, pivot_bal, pivot_recall],
    ['AUROC', 'Balanced Accuracy', 'Recall (Sensitivity)'],
    ['AUROC', 'Balanced Accuracy', 'Recall']
):
    pivot.plot(kind='bar', ax=ax, edgecolor='black', width=0.6)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    ax.set_xticklabels(pivot.index, rotation=25, ha='right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
    ax.legend(title='Model', loc='lower right', fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Fine-tuning Strategy Comparison — C2018 Test Set (PyTorch)', fontsize=14)
plt.tight_layout()
plt.savefig('finetune_pytorch_comparison_bars.png', bbox_inches='tight')
plt.show()

## Cell 19 — Threshold calibration (optional)

Both the TF and PyTorch pipelines use the default threshold τ=0.5, which produces
0% recall on the resistant class due to class imbalance pushing predicted probabilities
below 0.5. This cell demonstrates threshold recalibration using Youden's J statistic
on the **validation set** and reports the resulting sensitivity gain.

In [ ]:
from sklearn.metrics import roc_curve

def calibrate_threshold(
    model: nn.Module,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
    label: str = ''
):
    """
    Find the optimal decision threshold on the validation set using
    Youden's J statistic (J = Sensitivity + Specificity - 1),
    then evaluate on the test set with that threshold.
    """
    # Val probabilities
    val_probs  = get_predictions(model, X_val)
    fpr_v, tpr_v, thresholds_v = roc_curve(y_val, val_probs)

    # Youden's J: maximise sensitivity + specificity
    j_scores      = tpr_v - fpr_v
    best_idx      = np.argmax(j_scores)
    best_threshold = thresholds_v[best_idx]

    print(f"  Optimal threshold (Youden's J, val set) : {best_threshold:.4f}")
    print(f"  Val  — Sensitivity={tpr_v[best_idx]:.3f}  "
          f"Specificity={1 - fpr_v[best_idx]:.3f}")

    # Re-evaluate test set at calibrated threshold
    test_probs = get_predictions(model, X_test)
    y_pred_cal = (test_probs >= best_threshold).astype(int)

    cm  = confusion_matrix(y_test, y_pred_cal)
    bal = balanced_accuracy_score(y_test, y_pred_cal)

    print(f"\n  Test set @ τ={best_threshold:.4f}:")
    print(f"  Balanced accuracy : {bal:.4f}")
    print(f"  Confusion matrix  :\n{cm}")
    print(classification_report(
        y_test, y_pred_cal,
        target_names=['Susceptible (0)', 'Resistant (1)'],
        zero_division=0
    ))
    return best_threshold


# ── Run calibration for the best-performing model per organism ────────────────
# E. coli — best model was Full fine-tune
print("=" * 60)
print("Threshold calibration — E.coli / Full fine-tune")
ec_val_X, ec_val_y = build_splits(
    EC_META['species'], EC_META['drug'], verbose=False
)[3:5]  # returns (X_test, y_test) — we need val; re-extract below

# Re-extract validation arrays directly
def get_val_arrays(species, drug):
    mask = (labels_df['species'] == species) & (labels_df['drug'] == drug)
    sub  = labels_df[mask][['sample_id', 'response']].dropna()
    sub  = sub.merge(splits_df, on='sample_id', how='inner')
    sub  = sub[sub['sample_id'].isin(id_to_row)].copy()
    sub['row_idx'] = sub['sample_id'].map(id_to_row)
    rows = sub[sub['Set'] == 'validation']
    X    = raw_spectra[rows['row_idx'].values].astype(np.float32)
    y    = rows['response'].values.astype(np.float32)
    return X, y

ec_X_val, ec_y_val = get_val_arrays(EC_META['species'], EC_META['drug'])
kp_X_val, kp_y_val = get_val_arrays(KP_META['species'], KP_META['drug'])
sa_X_val, sa_y_val = get_val_arrays(SA_META['species'], SA_META['drug'])

print("\n" + "=" * 60)
print("E.coli / Full fine-tune")
calibrate_threshold(ec_ft_full, ec_X_val, ec_y_val, ec_X_test, ec_y_test,
                    label='E.coli/Ceftriaxone — Full FT calibrated')

print("\n" + "=" * 60)
print("K.pneumoniae / Baseline")
calibrate_threshold(kp_baseline, kp_X_val, kp_y_val, kp_X_test, kp_y_test,
                    label='K.pneumoniae/Ceftriaxone — Baseline calibrated')

print("\n" + "=" * 60)
print("S.aureus / Baseline")
calibrate_threshold(sa_baseline, sa_X_val, sa_y_val, sa_X_test, sa_y_test,
                    label='S.aureus/Oxacillin — Baseline calibrated')

---
## Notebook Summary

| Step | Tool / Library | Purpose |
|---|---|---|
| Load `.h5` | TensorFlow / Keras | Read pretrained MSDeepAMR weights |
| Export to ONNX | `tf2onnx` (opset 13) | Framework-neutral weight representation |
| Convert to PyTorch | `onnx2torch` | Reconstruct as trainable `nn.Module` |
| Data pipeline | `torch.utils.data` | Dataset + WeightedRandomSampler |
| Fine-tuning | PyTorch + Adam | Identical LRs to TF notebook |
| Callbacks | Custom `EarlyStopping` + `ReduceLROnPlateau` | Prevent overfitting |
| Evaluation | scikit-learn | AUROC, AUPRC, balanced accuracy |
| Calibration | Youden's J | Unlock true clinical sensitivity |

### Key differences vs. the TensorFlow notebook

1. **WeightedRandomSampler** — the PyTorch DataLoader oversamples the Resistant class
   during training, which directly addresses the class-imbalance problem that caused
   0% recall in the TF version.
2. **No metric-logging bug** — PyTorch computes all metrics from raw numpy arrays via
   scikit-learn; there is no equivalent of the Keras 3 metric-state initialisation issue.
3. **Portable weights** — fine-tuned models are saved as `.pt` state-dicts and can be
   loaded independently of TensorFlow.
4. **Threshold calibration** included inline as a first-class experiment step.